In [ ]:
# ============================================================
# ASSIGNMENT 2 - A2_03 RESUME: continue NRMS from pushed checkpoint (MIND, LARGE)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Loads mind_nrms_best.pt (epoch-0, dev AUC 0.5712) from HF and continues training with:
#   - lr LOWERED to 1e-4 (1e-3 was too high -> loss was rising)
#   - gradient clipping (max-norm 1.0)
#   - a mid-epoch DIVERGENCE GUARD (auto-stops if loss climbs vs epoch start)
# Trains 1 more epoch, evaluates dev, and if it improves pushes a NEW best checkpoint.
# If loss still doesn't descend -> we keep the 0.5712 baseline and move on.
#
# NEEDS: Kaggle MIND-large mounted. Loads checkpoint from HF donbosoc/mind-artifacts.
# ============================================================
!pip install huggingface_hub -q
import os, glob, re, math, numpy as np, pandas as pd, random, time
import torch, torch.nn as nn, torch.nn.functional as Fnn
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient
random.seed(0); torch.manual_seed(0)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:",device)
HF_REPO="donbosoc/mind-artifacts"
tok=UserSecretsClient().get_secret("HF_TOKEN"); api=HfApi(token=tok)
LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]

# ---- resume config ----
RESUME_CKPT = "mind_nrms_best.pt"   # from the interrupted run
NEW_LR      = 1e-4                  # lowered
GRAD_CLIP   = 1.0
EPOCHS_MORE = 1
BATCH=64; NPRATIO=4; MAX_TITLE=20; MAX_HIST=30


In [ ]:
# ---- rebuild vocab/titles EXACTLY as the original (same order -> same ids) ----
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","title"]) for d in (LTR,LDV)]).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna("")
title_tokens={pfx(r.news_id):tok(r.title)[:MAX_TITLE] for r in news.itertuples()}
vocab={"<pad>":0,"<unk>":1}
for tks in title_tokens.values():
    for w in tks:
        if w not in vocab: vocab[w]=len(vocab)
art_ids=[pfx(r.news_id) for r in news.itertuples()]; art_row={a:i for i,a in enumerate(art_ids)}
def enc_title(aid):
    idx=[vocab.get(w,1) for w in title_tokens.get(aid,[])][:MAX_TITLE]
    return idx+[0]*(MAX_TITLE-len(idx))
title_idx=np.array([enc_title(a) for a in art_ids],dtype=np.int64)
print("vocab:",len(vocab),"| titles:",title_idx.shape)


In [ ]:
# ---- NRMS (same architecture) + load checkpoint from HF ----
class AdditiveAttention(nn.Module):
    def __init__(s,dim,hid=200):
        super().__init__(); s.p=nn.Linear(dim,hid); s.q=nn.Linear(hid,1,bias=False)
    def forward(s,x,mask=None):
        e=torch.tanh(s.p(x)); a=s.q(e).squeeze(-1)
        if mask is not None: a=a.masked_fill(mask==0,-1e9)
        return (x*torch.softmax(a,-1).unsqueeze(-1)).sum(1)
class NewsEncoder(nn.Module):
    def __init__(s,V,ed=300,h=15,hd=20):
        super().__init__(); d=h*hd
        s.emb=nn.Embedding(V,ed,padding_idx=0); s.attn=nn.MultiheadAttention(ed,h,batch_first=True)
        s.proj=nn.Linear(ed,d); s.add=AdditiveAttention(d); s.drop=nn.Dropout(0.2)
    def forward(s,t):
        m=(t!=0); x=s.drop(s.emb(t)); x,_=s.attn(x,x,x,key_padding_mask=~m)
        x=s.drop(x); x=torch.relu(s.proj(x)); return s.add(x,m)
class UserEncoder(nn.Module):
    def __init__(s,d):
        super().__init__(); s.attn=nn.MultiheadAttention(d,5,batch_first=True); s.add=AdditiveAttention(d)
    def forward(s,hv,mask):
        x,_=s.attn(hv,hv,hv,key_padding_mask=~mask); return s.add(x,mask)
class NRMS(nn.Module):
    def __init__(s,V,h=15,hd=20):
        super().__init__(); s.news=NewsEncoder(V,h=h,hd=hd); s.user=UserEncoder(h*hd)
    def forward(s,ht,hm,ct):
        B,H,L=ht.shape; C=ct.shape[1]
        hv=s.news(ht.reshape(B*H,L)).reshape(B,H,-1); u=s.user(hv,hm)
        cv=s.news(ct.reshape(B*C,L)).reshape(B,C,-1)
        return torch.bmm(cv,u.unsqueeze(-1)).squeeze(-1)

model=NRMS(len(vocab)).to(device)
ckpt_path=hf_hub_download(HF_REPO, RESUME_CKPT, repo_type="dataset", token=tok)
ck=torch.load(ckpt_path, map_location=device)
# vocab-size guard: checkpoint must match current vocab
saved_V = ck["model"]["news.emb.emb.weight"].shape[0] if "news.emb.emb.weight" in ck["model"] else None
try:
    model.load_state_dict(ck["model"]); print("loaded checkpoint:", RESUME_CKPT)
except Exception as e:
    print("state_dict load issue:", e)
    print("saved vocab size:", ck.get("vocab") and len(ck["vocab"]), "| current:", len(vocab))
    raise


In [ ]:
# ---- training samples + dev eval set (same as original) ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
def make_samples(b_df):
    S=[]
    for u,imps in zip(b_df["user_id"],b_df["impressions"]):
        if not isinstance(imps,str): continue
        uid=pfx(u); hist=hist_lut.get(uid)
        if not hist: continue
        pos=[pfx(t.split("-")[0]) for t in imps.split() if t.endswith("-1")]
        neg=[pfx(t.split("-")[0]) for t in imps.split() if t.endswith("-0")]
        if not pos or not neg: continue
        for p in pos:
            ng=random.sample(neg,NPRATIO) if len(neg)>=NPRATIO else neg+[random.choice(neg) for _ in range(NPRATIO-len(neg))]
            S.append((uid,p,ng))
    return S
train_samples=make_samples(b_tr); print("train samples:",len(train_samples))
def hist_tensor(uid):
    h=hist_lut.get(uid,[])[-MAX_HIST:]
    rows=[art_row[a] for a in h if a in art_row][:MAX_HIST]
    mask=[1]*len(rows)+[0]*(MAX_HIST-len(rows)); rows=rows+[0]*(MAX_HIST-len(rows))
    return rows,mask
def collate(batch):
    H=[];HM=[];C=[]
    for uid,pos,negs in batch:
        hr,hm=hist_tensor(uid); H.append(hr);HM.append(hm); C.append([art_row.get(c,0) for c in [pos]+negs])
    H=np.array(H);C=np.array(C)
    return (torch.tensor(title_idx[H]),torch.tensor(np.array(HM),dtype=torch.bool),
            torch.tensor(title_idx[C]),torch.zeros(len(batch),dtype=torch.long))

def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def _mrr(s,lb):
    for rank,idx in enumerate(np.argsort(-s),1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def _ndcg(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(np.sort(lb)[::-1][:k]))
    return float(dcg/idcg) if idcg>0 else 0.0
dev_impr=[]
for iid,u,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["impressions"]):
    if not isinstance(imps,str): continue
    uid=pfx(u)
    if not hist_lut.get(uid): continue
    cand=[pfx(t.split("-")[0]) for t in imps.split()]
    labs=np.array([1 if t.endswith("-1") else 0 for t in imps.split()])
    if labs.sum()>0: dev_impr.append((iid,uid,cand,labs))
random.Random(0).shuffle(dev_impr); dev_impr=dev_impr[:40000]
@torch.no_grad()
def eval_nrms():
    model.eval(); A=[];M=[];N5=[];N10=[]
    for iid,uid,cand,labs in dev_impr:
        hr,hm=hist_tensor(uid)
        ht=torch.tensor(title_idx[np.array(hr)]).unsqueeze(0).to(device)
        hmask=torch.tensor(hm,dtype=torch.bool).unsqueeze(0).to(device)
        ct=torch.tensor(title_idx[np.array([art_row.get(c,0) for c in cand])]).unsqueeze(0).to(device)
        s=model(ht,hmask,ct).squeeze(0).cpu().numpy(); a=_auc(s,labs)
        if a is not None: A.append(a)
        M.append(_mrr(s,labs));N5.append(_ndcg(s,labs,5));N10.append(_ndcg(s,labs,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))
print("dev eval set:", len(dev_impr))
print("baseline (loaded ckpt) dev:", {k:round(v,4) for k,v in eval_nrms().items()})


In [ ]:
# ---- resume training: lr=1e-4 + grad clip + mid-epoch divergence guard ----
from torch.utils.data import DataLoader
opt=torch.optim.Adam(model.parameters(), lr=NEW_LR)
loader=DataLoader(train_samples,batch_size=BATCH,shuffle=True,collate_fn=collate,num_workers=2)

def push(tag):
    p=f"/kaggle/working/mind_nrms_{tag}.pt"; torch.save({"model":model.state_dict(),"vocab":vocab},p)
    try: api.upload_file(path_or_fileobj=p,path_in_repo=f"mind_nrms_{tag}.pt",repo_id=HF_REPO,repo_type="dataset"); print("  pushed",tag)
    except Exception as e: print("  HF push failed:",e)

prev_best=eval_nrms()["AUC"]; print(f"resuming from dev AUC {prev_best:.4f} with lr={NEW_LR}")
for ep in range(EPOCHS_MORE):
    model.train(); t0=time.time(); tot=0; nb=0; run=[]
    for ht,hm,ct,y in loader:
        ht,hm,ct,y=ht.to(device),hm.to(device),ct.to(device),y.to(device)
        loss=Fnn.cross_entropy(model(ht,hm,ct),y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)   # grad clip
        opt.step(); tot+=loss.item(); nb+=1; run.append(loss.item())
        if nb%2000==0:
            recent=np.mean(run[-2000:]); early=np.mean(run[:2000])
            print(f"  step {nb}/{len(loader)} loss {tot/nb:.4f} (recent2k {recent:.4f})")
            # DIVERGENCE GUARD: if well past warmup and recent loss is worse than the first 2k, bail
            if nb>=10000 and recent > early + 0.02:
                print(f"  DIVERGENCE GUARD: recent loss {recent:.4f} > start {early:.4f}+0.02 -> stopping epoch")
                break
    m=eval_nrms()
    print(f"resume epoch {ep}: train_loss {tot/max(1,nb):.4f} | dev AUC {m['AUC']:.4f} "
          f"MRR {m['MRR']:.4f} nDCG@10 {m['nDCG10']:.4f} ({(time.time()-t0)/60:.1f} min)")
    push(f"resume_ep{ep}")
    if m["AUC"]>prev_best:
        prev_best=m["AUC"]; push("best"); print(f"  improved -> new best {prev_best:.4f}, pushed as best")
    else:
        print(f"  no improvement over {prev_best:.4f}; keeping previous best. lr=1e-4 didn't help -> use 0.5712 baseline.")
print("done. best dev AUC:", round(prev_best,4))
